In [ ]:
import subprocess, sys

def _require(package, import_name=None):
    """Install package if not already importable."""
    name = import_name or package
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

_require("pandas")
_require("numpy")
_require("matplotlib")
_require("mkl")
_require("earthengine-api", "ee")
_require("geemap")
_require("scikit-learn", "sklearn")
_require("rasterio")
_require("torchvision")
_require("pickle5", "pickle")   # no-op on Python 3.8+ where pickle is built-in

import os, math, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import mkl
    mkl.set_num_threads(2)
except ImportError:
    pass

import ee
import geemap

from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn import datasets

import rasterio

try:
    from torchvision.transforms import ToTensor
except ImportError:
    pass   # torchvision optional — not used in core pipeline

#######################################################################
# GLOBAL CONSTANTS
#######################################################################
CONFIDENCE_LEVEL  = 80           # CDL confidence threshold (0–100 scale)
NDVI_THRESHOLD    = 0.25

SPECIFIC_CROPS = [2.0, 36.0, 69.0, 75.0, 204.0, 500.0]
NUM_CLASSES    = len(SPECIFIC_CROPS)

TRAIN_YEAR        = 2021         # year used for training data
PRED_YEAR         = 2022         # year used for prediction / validation
NUM_SAMPLES_TRAIN = 100000
NUM_SAMPLES_PRED  = 200000

# Set to an integer to cap each class at that many samples,
# or None to automatically undersample to the smallest class size.
SAMPLES_PER_CLASS = None

# LABEL_COL is set dynamically in the GEE setup cell below

In [ ]:
#######################################################################
# GEE INITIALISATION & SETUP
#######################################################################

ee.Initialize(project='ee-rmccormick314-gfsad')

# Cal Area 3 bounding box  [west, south, east, north]
TRAIN_AOI = ee.Geometry.Rectangle([-120.6333, 36.9333, -120.3667, 37.0667])
PRED_AOI  = ee.Geometry.Rectangle([-120.6333, 36.9333, -120.3667, 37.0667])

# ── Training Year Data (TRAIN_YEAR) ───────────────────────────────────
train_embedding = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f'{TRAIN_YEAR}-01-01', f'{TRAIN_YEAR}-12-31')
    .filterBounds(TRAIN_AOI)
    .first()
)
train_cdl = (
    ee.ImageCollection("USDA/NASS/CDL")
    .filterDate(f'{TRAIN_YEAR}-01-01', f'{TRAIN_YEAR}-12-31')
    .first()
)

# ── Prediction Year Data (PRED_YEAR) ──────────────────────────────────
pred_embedding = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f'{PRED_YEAR}-01-01', f'{PRED_YEAR}-12-31')
    .filterBounds(PRED_AOI)
    .first()
)
pred_cdl = (
    ee.ImageCollection("USDA/NASS/CDL")
    .filterDate(f'{PRED_YEAR}-01-01', f'{PRED_YEAR}-12-31')
    .first()
)

# ── Derive LABEL_COL from training embedding dimensionality ───────────
EMBEDDING_BANDS     = train_embedding.bandNames().getInfo()
NUM_EMBEDDING_BANDS = len(EMBEDDING_BANDS)
LABEL_COL           = NUM_EMBEDDING_BANDS   # label column sits right after embedding bands

print(f"Train year: {TRAIN_YEAR}  |  Pred year: {PRED_YEAR}")
print(f"Satellite Embedding bands ({NUM_EMBEDDING_BANDS}): {EMBEDDING_BANDS}")
print(f"LABEL_COL set to: {LABEL_COL}")

In [ ]:
#######################################################################
# SAMPLE TRAINING DATA FROM GEE  (TRAIN_YEAR)
#######################################################################

training_image = (
    train_embedding
    .addBands(train_cdl.select('cropland').rename('cdl_label'))
    .addBands(train_cdl.select('confidence').rename('cdl_confidence'))
)

print(f"Sampling {NUM_SAMPLES_TRAIN} training pixels from GEE (year={TRAIN_YEAR}) ...")
training_fc = training_image.sample(
    region    = TRAIN_AOI,
    scale     = 10,
    numPixels = NUM_SAMPLES_TRAIN,
    seed      = 42,
    geometries= False,
    dropNulls = True
)

# Convert FeatureCollection → DataFrame
# remove_geom=False avoids a pandas compatibility bug in geemap's drop() call
DESIS_df = geemap.ee_to_df(training_fc, remove_geom=False)
DESIS_df = DESIS_df.drop(columns=[c for c in ['geo'] if c in DESIS_df.columns])

# Rename columns: embedding band names → integer indices, label/confidence → LABEL_COL / LABEL_COL+1
col_map = {b: i for i, b in enumerate(EMBEDDING_BANDS)}
col_map['cdl_label']      = LABEL_COL
col_map['cdl_confidence'] = LABEL_COL + 1
DESIS_df = DESIS_df.rename(columns=col_map)

print(f"Recovered {len(DESIS_df)} training samples.")
print(DESIS_df.head())

In [ ]:
#######################################################################
# PROCESS TRAINING DATA
#######################################################################

desis_df = DESIS_df.copy()

# CDL label remapping
desis_df[LABEL_COL] = np.where(desis_df[LABEL_COL] == 225.0, 1.0,   desis_df[LABEL_COL])
desis_df[LABEL_COL] = np.where(desis_df[LABEL_COL] == 24.0,  1.0,   desis_df[LABEL_COL])
desis_df[LABEL_COL] = np.where(desis_df[LABEL_COL] == 75.0,  204.0, desis_df[LABEL_COL])

# Collapse non-target crops → 'Other' (500)
desis_df[LABEL_COL] = np.where(
    ~desis_df[LABEL_COL].isin(SPECIFIC_CROPS), 500.0, desis_df[LABEL_COL]
)

# Filter by CDL confidence (LABEL_COL+1 holds CDL confidence on 0–100 scale)
desis_df = desis_df[desis_df[LABEL_COL + 1] >= CONFIDENCE_LEVEL]
desis_df = desis_df.drop(LABEL_COL + 1, axis=1)
desis_df = desis_df.reset_index(drop=True)

# Balance classes by undersampling — use concat to avoid pandas 2.x groupby.apply
# dropping the groupby column from the result
n_per_class = SAMPLES_PER_CLASS if SAMPLES_PER_CLASS is not None \
              else int(desis_df[LABEL_COL].value_counts().min())
desis_df = pd.concat([
    grp.sample(min(len(grp), n_per_class), random_state=42)
    for _, grp in desis_df.groupby(LABEL_COL)
]).reset_index(drop=True)

print(f"Balanced to {n_per_class} samples per class.")
print(desis_df[LABEL_COL].value_counts())

desis_df_all = desis_df

# Feature / label split
FEATURES = desis_df.loc[:, desis_df.columns != LABEL_COL]

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer
imp = SimpleImputer(missing_values=np.nan, strategy='mean')
imp.fit(FEATURES)
FEATURES = imp.transform(FEATURES)

LABELS = desis_df[LABEL_COL]

X_train, X_test, y_train, y_test = train_test_split(
    FEATURES, LABELS,
    test_size    = 0.01,
    random_state = 42,
    stratify     = desis_df[LABEL_COL]
)

print("\nDISTRIBUTION:")
print(pd.DataFrame(y_train).value_counts(normalize=True) * 100)

In [ ]:
desis_df[LABEL_COL].value_counts()

204.0    306566
500.0    232155
2.0       73277
36.0      63742
69.0      35383
Name: 86, dtype: int64

In [ ]:
#######################################################################
# BEGIN: ESTABLISH AND FIT DATA SCALER
#######################################################################

SCALER_NAME = '4-1-24-Scaler'

try:
    # If scaler exists, open it and fit the current data to it
    scaler = pickle.load(open(SCALER_NAME + ".pickle", "rb"))
    scaler.fit(X_train)
except (OSError, IOError) as e:
    # If scaler does NOT exist, create a new one, fit data, and save
    #pickle.dump(normalizer, open("clf_normalizer.pickle", "wb"))
    scaler = StandardScaler() 
    scaler.fit(X_train)
    pickle.dump(scaler, open(SCALER_NAME + ".pickle", "wb"))

# Establish train / test sets & scale

X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

#######################################################################
# END: SCALING
#######################################################################

In [ ]:
#######################################################################
# BEGIN: CREATE/LOAD/FIT MODEL
#######################################################################

MODEL_NAME = "4-2-24_model"

try:
    # If model exists, try to load it
    clf = pickle.load( open( MODEL_NAME + ".pickle", "rb" ) )
    check_is_fitted = True  # will raise NotFittedError below if invalid
    from sklearn.utils.validation import check_is_fitted
    check_is_fitted(clf)
    print("Loaded existing model from", MODEL_NAME + ".pickle")

except (OSError, IOError):
    clf = None  # file not found — fall through to training

except Exception:
    print("Pickle exists but model is not fitted — retraining.")
    clf = None

if clf is None:
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier

    base_clf = MLPClassifier(
        solver             = 'adam',
        activation         = 'relu',
        alpha              = 0.0001,
        hidden_layer_sizes = ( math.floor( LABEL_COL / 2 ) ),
        learning_rate      = 'constant',
        max_iter           = 100,
        random_state       = 69420,
        warm_start         = True,
        verbose            = True,
        shuffle            = True,
        early_stopping     = True,
        validation_fraction = 0.1
    )

    parameter_space = {
        'hidden_layer_sizes': [ ( LABEL_COL ), ( LABEL_COL*2 ), ( math.floor( LABEL_COL / 2 ) ) ],
        'max_iter': [ 5, 10, 25, 50, 100, 150 ],
        'solver': ['sgd', 'adam', 'lbfgs'],
    }

    cv = GridSearchCV(base_clf, parameter_space, n_jobs=-1, cv=3)
    history = cv.fit(X_train, y_train)

    clf = cv.best_estimator_

    pickle.dump(clf, open(MODEL_NAME + ".pickle", "wb"))
    pickle.dump(history, open(MODEL_NAME + "_Loss.pickle", "wb"))

    print('Best parameters found:\n', cv.best_params_)

#######################################################################
# END: MODEL ESTABLISHMENT & TRAINING
#######################################################################

In [ ]:
#######################################################################
# SAMPLE PREDICTION DATA FROM GEE (PRED_YEAR) AND RUN INITIAL CLASSIFICATION
#######################################################################

print(f"Sampling {NUM_SAMPLES_PRED} prediction pixels from GEE (year={PRED_YEAR}) ...")

pred_image = (
    pred_embedding
    .addBands(pred_cdl.select('cropland').rename('cdl_label'))
)

pred_fc = pred_image.sample(
    region    = PRED_AOI,
    scale     = 10,
    numPixels = NUM_SAMPLES_PRED,
    seed      = 0,
    geometries= True,    # retain coordinates for spatial output
    dropNulls = True
)

# remove_geom=False avoids a pandas compatibility bug in geemap's drop() call
pred_df = geemap.ee_to_df(pred_fc, remove_geom=False)

# Rename columns
pred_col_map = {b: i for i, b in enumerate(EMBEDDING_BANDS)}
pred_col_map['cdl_label'] = LABEL_COL
pred_df = pred_df.rename(columns=pred_col_map)

# CDL label remapping (same as training)
pred_df[LABEL_COL] = np.where(pred_df[LABEL_COL] == 225.0, 1.0,   pred_df[LABEL_COL])
pred_df[LABEL_COL] = np.where(pred_df[LABEL_COL] == 24.0,  1.0,   pred_df[LABEL_COL])
pred_df[LABEL_COL] = np.where(pred_df[LABEL_COL] == 75.0,  204.0, pred_df[LABEL_COL])
pred_df[LABEL_COL] = np.where(
    ~pred_df[LABEL_COL].isin(SPECIFIC_CROPS), 500.0, pred_df[LABEL_COL]
)

desis_df_all = pred_df.copy()
labels_all   = pred_df[LABEL_COL].values

# Prepare features (embedding columns only)
feature_cols = list(range(NUM_EMBEDDING_BANDS))
imp.fit(pred_df[feature_cols])
train_all = imp.transform(pred_df[feature_cols])
train_all = scaler.transform(train_all)

predictions = clf.predict(train_all)

print("\nPREDICTION DISTRIBUTION:")
print(pd.DataFrame(predictions).value_counts(normalize=True) * 100)
print("\nACTUAL (CDL) DISTRIBUTION:")
print(pd.DataFrame(labels_all).value_counts(normalize=True) * 100)
print("\nAccuracy:", str(accuracy_score(predictions, labels_all) * 100) + "%")

In [ ]:
#######################################################################
# BEGIN: CREATE PREDICTION IMAGE ERROR MATRIX
#######################################################################
SPECIFIC_CROPS = [ 2.0, 36.0, 69.0, 204.0, 500.0 ]

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

desis_confusion_matrix = confusion_matrix(labels_all, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=desis_confusion_matrix,
                              display_labels=SPECIFIC_CROPS)

confusion_df = pd.DataFrame(desis_confusion_matrix, columns=SPECIFIC_CROPS, index=SPECIFIC_CROPS)
confusion_df = confusion_df.astype(object)

confusion_df.loc['Total', :] = confusion_df.sum(axis=0)
confusion_df.loc[:, 'Total'] = confusion_df.sum(axis=1)

# Pre-compute overall accuracy from numeric values before mixing in strings
overall_acc = str(round(
    sum(int(confusion_df.loc[i, i]) for i in SPECIFIC_CROPS)
    / int(confusion_df.loc['Total', 'Total']) * 100, 2)) + "%"

# User Accuracy column
confusion_df['User Acc'] = pd.Series(dtype=object, index=confusion_df.index)
for index in SPECIFIC_CROPS:
    confusion_df.loc[index, 'User Acc'] = str(round(
        int(confusion_df.loc[index, index]) / int(confusion_df.loc[index, 'Total']) * 100, 2)) + "%"
confusion_df.loc['Total', 'User Acc'] = None

# Producer Accuracy row — initialize with None, then fill strings
confusion_df.loc['Producer Acc', :] = None
for index in SPECIFIC_CROPS:
    confusion_df.loc['Producer Acc', index] = str(round(
        int(confusion_df.loc[index, index]) / int(confusion_df.loc['Total', index]) * 100, 2)) + "%"
confusion_df.loc['Producer Acc', 'User Acc'] = overall_acc

confusion_df = confusion_df.rename(
    columns={
        75.0:"Almonds", 204.0:"Pistachios", 2.0:"Cotton", 36.0:"Alfalfa",
        225.0:"Dbl Crop WinWht/Corn", 54.0:"Tomatoes", 69.0:"Grapes",
        24.0:"Winter Wheat", 61.0:"Fallow", 1.0:"Corn", 500.0:"Other",
        "Total":"Total", "User Acc":"User Accuracy"
    },
    index={
        75.0:"Almonds", 204.0:"Pistachios", 2.0:"Cotton", 36.0:"Alfalfa",
        225.0:"Dbl Crop WinWht/Corn", 54.0:"Tomatoes", 69.0:"Grapes",
        24.0:"Winter Wheat", 61.0:"Fallow", 1.0:"Corn", 500.0:"Other",
        "Total":"Total", "Producer Acc":"Producer Accuracy"
    }
)

confusion_df.to_csv(MODEL_NAME + "_accuracy_scores.csv")
confusion_df

#######################################################################
# END: PREDICTION IMAGE ERROR MATRIX
#######################################################################

In [ ]:
# Snapshot of initial clf predictions before grain sub-classification
unmod_df = pred_df.copy()
unmod_df[LABEL_COL] = predictions

In [ ]:
SECONDARY_CROPS = [1.0, 24.0, 225.0]

grain_df = DESIS_df.copy()
grain_df = grain_df.loc[grain_df[LABEL_COL].isin(SECONDARY_CROPS)]
grain_df = grain_df.drop(LABEL_COL + 1, axis=1)   # drop CDL confidence
grain_df = grain_df.reset_index(drop=True)

grain_train = scaler.transform(imp.transform(grain_df.loc[:, grain_df.columns != LABEL_COL]))
grain_label = grain_df[LABEL_COL]

grain_clf = MLPClassifier(
    solver             = 'adam',
    activation         = 'relu',
    alpha              = 0.0001,
    hidden_layer_sizes = (math.floor(LABEL_COL / 2),),
    learning_rate      = 'constant',
    max_iter           = 100,
    random_state       = 69420,
    warm_start         = True,
    verbose            = False,
    shuffle            = True,
    early_stopping     = True,
    validation_fraction= 0.1
)
grain_history = grain_clf.fit(grain_train, grain_label)


In [ ]:
desis_df_all[LABEL_COL] = predictions

new        = desis_df_all.copy()
le_grains  = new.loc[new[LABEL_COL].isin(SECONDARY_CROPS)]

grain_pred_features = le_grains[list(range(NUM_EMBEDDING_BANDS))]
grain_train_input   = scaler.transform(imp.transform(grain_pred_features))
grains_predict      = grain_clf.predict(grain_train_input)

desis_df_all.loc[desis_df_all[LABEL_COL].isin(SECONDARY_CROPS), LABEL_COL] = grains_predict


In [ ]:
predictions = desis_df_all[LABEL_COL]
desis_df_all[LABEL_COL].value_counts()

204.0    309918
500.0    161533
2.0       99607
36.0      70774
225.0     45293
24.0      41042
69.0      32290
1.0       12913
Name: 86, dtype: int64

In [ ]:
SECONDARY_CROPS = [75.0, 204.0]

grain_df = DESIS_df.copy()
grain_df = grain_df.loc[grain_df[LABEL_COL].isin(SECONDARY_CROPS)]
grain_df = grain_df.drop(LABEL_COL + 1, axis=1)   # drop CDL confidence
grain_df = grain_df.reset_index(drop=True)

grain_train = scaler.transform(imp.transform(grain_df.loc[:, grain_df.columns != LABEL_COL]))
grain_label = grain_df[LABEL_COL]

grain_clf = MLPClassifier(
    solver             = 'adam',
    activation         = 'relu',
    alpha              = 0.0001,
    hidden_layer_sizes = (math.floor(LABEL_COL / 2),),
    learning_rate      = 'constant',
    max_iter           = 100,
    random_state       = 69420,
    warm_start         = True,
    verbose            = False,
    shuffle            = True,
    early_stopping     = True,
    validation_fraction= 0.1
)
grain_history = grain_clf.fit(grain_train, grain_label)


In [ ]:
desis_df_all[LABEL_COL] = predictions

new       = desis_df_all.copy()
le_grains = new.loc[new[LABEL_COL].isin(SECONDARY_CROPS)]

grain_pred_features = le_grains[list(range(NUM_EMBEDDING_BANDS))]
grain_train_input   = scaler.transform(imp.transform(grain_pred_features))
grains_predict      = grain_clf.predict(grain_train_input)

desis_df_all.loc[desis_df_all[LABEL_COL].isin(SECONDARY_CROPS), LABEL_COL] = grains_predict


In [ ]:
FINAL_PREDICTIONS = desis_df_all[LABEL_COL]#.copy()

In [ ]:
desis_df_all[LABEL_COL].value_counts()

500.0    255863
75.0     227021
204.0    116263
2.0       86090
36.0      56094
69.0      32039
Name: 86, dtype: int64

In [ ]:
#######################################################################
# DOWNLOAD EMBEDDING FROM GEE AND CLASSIFY EVERY PIXEL
#######################################################################

import os
from sklearn.metrics import confusion_matrix, accuracy_score

SPECIFIC_CROPS    = [2.0, 36.0, 69.0, 75.0, 204.0, 500.0]
CROP_LABELS       = {2.0:'Cotton', 36.0:'Alfalfa', 69.0:'Grapes',
                     75.0:'Almonds', 204.0:'Pistachios', 500.0:'Other'}
OUT_DIR           = './Test_Max/'
os.makedirs(OUT_DIR, exist_ok=True)

embedding_tif  = OUT_DIR + f'{PRED_YEAR}_embedding.tif'
classified_tif = OUT_DIR + f'{PRED_YEAR}_classified.tif'

# ── 1. Download embedding (cached after first run) ────────────────────
if not os.path.exists(embedding_tif):
    print(f"Downloading {PRED_YEAR} embedding from GEE at 10 m resolution...")
    geemap.ee_export_image(
        pred_embedding,
        filename   = embedding_tif,
        scale      = 10,
        region     = PRED_AOI,
        file_per_band = False,
    )
    print("Download complete.")
else:
    print(f"Using cached embedding: {embedding_tif}")

# ── 2. Load raster ────────────────────────────────────────────────────
dataset      = rasterio.open(embedding_tif)
dataset_meta = dataset.meta
H, W         = dataset.shape
NUM_RASTER_BANDS = dataset.count
print(f"Raster size: {W} x {H} = {W*H:,} pixels  |  {NUM_RASTER_BANDS} bands")

pixels = np.array([dataset.read(b) for b in range(1, NUM_RASTER_BANDS + 1)]) \
           .reshape(NUM_RASTER_BANDS, -1).T        # shape: [H*W, bands]

# ── 3. Impute → scale → classify ─────────────────────────────────────
feat_scaled = scaler.transform(imp.transform(pixels[:, :NUM_EMBEDDING_BANDS]))

print(f"Classifying {H*W:,} pixels...")
raster_predictions = clf.predict(feat_scaled).astype(float)

# Re-classify tree-crop pixels (75 = almonds vs 204 = pistachios)
grain_mask = np.isin(raster_predictions, [75.0, 204.0])
if grain_mask.any():
    raster_predictions[grain_mask] = grain_clf.predict(
        feat_scaled[grain_mask]).astype(float)

# ── 4. Print distribution ─────────────────────────────────────────────
unique, counts = np.unique(raster_predictions, return_counts=True)
print("\nClassification distribution:")
for u, c in zip(unique, counts):
    print(f"  {CROP_LABELS.get(u, str(u))}: {c:,} px  ({c/len(raster_predictions)*100:.1f}%)")

In [ ]:
labels_all.value_counts()

500.0    272710
75.0     231329
2.0       87302
204.0     82351
36.0      63835
69.0      35843
Name: 86, dtype: int64

In [ ]:
FINAL_PREDICTIONS.value_counts()

500.0    255863
75.0     227021
204.0    116263
2.0       86090
36.0      56094
69.0      32039
Name: 86, dtype: int64

In [ ]:
#######################################################################
# WRITE FINAL PREDICTIONS TO CSV
#######################################################################

labels_all = pred_df[LABEL_COL].values
predictions = FINAL_PREDICTIONS.values

# Attach final predictions to the prediction DataFrame (includes .geo coordinates)
pred_df['prediction'] = FINAL_PREDICTIONS.values
pred_df.to_csv(MODEL_NAME + '_CLASSIFIED.csv', index=False)
print(f"Final predictions saved to {MODEL_NAME}_CLASSIFIED.csv")

print("\nFINAL PREDICTION DISTRIBUTION:")
print(pd.DataFrame(predictions).value_counts(normalize=True) * 100)
print("\nACTUAL (CDL) DISTRIBUTION:")
print(pd.DataFrame(labels_all).value_counts(normalize=True) * 100)
print("\nAccuracy:", str(accuracy_score(predictions, labels_all) * 100) + "%")


In [ ]:
FINAL_PREDICTIONS.value_counts()

75.0     287215
500.0    161533
2.0       99607
36.0      70774
225.0     45293
24.0      41042
69.0      32290
204.0     22703
1.0       12913
Name: 86, dtype: int64

In [ ]:
grain_final_thing = pd.DataFrame(DESIS_df[LABEL_COL])
grain_final_thing[1] = FINAL_PREDICTIONS

grain_final_thing = grain_final_thing.loc[grain_final_thing[1].isin( SECONDARY_CROPS ) ]

grain_final_thing[LABEL_COL] = np.where(grain_final_thing[LABEL_COL] == 75.0, 204.0, grain_final_thing[LABEL_COL] )

# Set all crop labels NOT in list of interested crops to 'Other Crops' (500)
SPECIFIC_CROPS = [ 1.0, 24.0, 2.0, 36.0, 69.0, 75.0, 204.0, 225.0, 500.0 ]

grain_final_thing[LABEL_COL] = np.where(~grain_final_thing[LABEL_COL].isin(SPECIFIC_CROPS), 500, grain_final_thing[LABEL_COL])

grain_final_thing

,86,1
19,24.0,1.0
20,24.0,1.0
21,24.0,1.0
22,24.0,1.0
23,24.0,1.0
...,...,...
772948,500.0,1.0
772949,500.0,1.0
772950,500.0,1.0
772951,500.0,1.0


In [ ]:
#######################################################################
# BEGIN: CREATE PREDICTION IMAGE ERROR MATRIX (ROUND 2)
#######################################################################

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

SPECIFIC_CROPS = [ 1.0, 24.0, 2.0, 36.0, 69.0, 204.0, 225.0, 500.0 ]

labels_all  = grain_final_thing[LABEL_COL]
predictions = grain_final_thing[1]

desis_confusion_matrix = confusion_matrix(predictions, labels_all)
disp = ConfusionMatrixDisplay(confusion_matrix=desis_confusion_matrix,
                              display_labels=SPECIFIC_CROPS)

confusion_df = pd.DataFrame(desis_confusion_matrix, columns=SPECIFIC_CROPS, index=SPECIFIC_CROPS)
confusion_df = confusion_df.astype(object)

confusion_df.loc['Total', :] = confusion_df.sum(axis=0)
confusion_df.loc[:, 'Total'] = confusion_df.sum(axis=1)

# Pre-compute overall accuracy from numeric values before mixing in strings
overall_acc = str(round(
    sum(int(confusion_df.loc[i, i]) for i in SPECIFIC_CROPS)
    / int(confusion_df.loc['Total', 'Total']) * 100, 2)) + "%"

# User Accuracy column
confusion_df['User Acc'] = pd.Series(dtype=object, index=confusion_df.index)
for index in SPECIFIC_CROPS:
    confusion_df.loc[index, 'User Acc'] = str(round(
        int(confusion_df.loc[index, index]) / int(confusion_df.loc[index, 'Total']) * 100, 2)) + "%"
confusion_df.loc['Total', 'User Acc'] = None

# Producer Accuracy row — initialize with None, then fill strings
confusion_df.loc['Producer Acc', :] = None
for index in SPECIFIC_CROPS:
    confusion_df.loc['Producer Acc', index] = str(round(
        int(confusion_df.loc[index, index]) / int(confusion_df.loc['Total', index]) * 100, 2)) + "%"
confusion_df.loc['Producer Acc', 'User Acc'] = overall_acc

confusion_df = confusion_df.rename(
    columns={
        75.0:"Almonds", 204.0:"Tree Crops", 2.0:"Cotton", 36.0:"Alfalfa",
        225.0:"Dbl Crop WinWht/Corn", 54.0:"Tomatoes", 69.0:"Grapes",
        24.0:"Winter Wheat", 61.0:"Fallow", 1.0:"Corn", 500.0:"Other",
        "Total":"Total", "User Acc":"User Accuracy"
    },
    index={
        75.0:"Almonds", 204.0:"Tree Crops", 2.0:"Cotton", 36.0:"Alfalfa",
        225.0:"Dbl Crop WinWht/Corn", 54.0:"Tomatoes", 69.0:"Grapes",
        24.0:"Winter Wheat", 61.0:"Fallow", 1.0:"Corn", 500.0:"Other",
        "Total":"Total", "Producer Acc":"Producer Accuracy"
    }
)

confusion_df.to_csv(MODEL_NAME + "_accuracy_scores_ROUND2.csv")
confusion_df

#######################################################################
# END: PREDICTION IMAGE ERROR MATRIX (ROUND 2)
#######################################################################

In [ ]:
#######################################################################
# WRITE CLASSIFIED IMAGE TO GEOTIFF + PREVIEW
#######################################################################

out_meta = dataset_meta.copy()
out_meta.update({'count': 1, 'dtype': 'float32', 'nodata': -1})

with rasterio.open(classified_tif, 'w', **out_meta) as dst:
    dst.write(raster_predictions.reshape(H, W).astype('float32'), 1)

print(f"Saved: {classified_tif}")
print(f"  Size : {W} x {H} pixels")
print(f"  CRS  : {dataset_meta.get('crs')}")

# Color-mapped preview
fig, ax = plt.subplots(figsize=(9, 7))
cmap    = plt.cm.get_cmap('tab10', len(SPECIFIC_CROPS))
val_idx = {v: i for i, v in enumerate(sorted(SPECIFIC_CROPS))}
indexed = np.vectorize(val_idx.get)(raster_predictions).reshape(H, W)
ax.imshow(indexed, cmap=cmap, vmin=0, vmax=len(SPECIFIC_CROPS) - 1, interpolation='nearest')
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap(val_idx[v])) for v in sorted(SPECIFIC_CROPS)]
ax.legend(handles, [CROP_LABELS.get(v, str(v)) for v in sorted(SPECIFIC_CROPS)],
          loc='lower right', fontsize=9)
ax.set_title(f'Predicted Crop Classification — {PRED_YEAR}  (trained on {TRAIN_YEAR})')
ax.axis('off')
plt.tight_layout()
preview_path = classified_tif.replace('.tif', '_preview.png')
plt.savefig(preview_path, dpi=150)
plt.show()
print(f"Preview saved: {preview_path}")